# NYC TLC Yellow Taxi Operational Foundations Pipeline
## From Raw Telemetry to Dependable Business Decisions (Classes 4–8)

**Track B:** NYC Taxi & Limousine Commission (TLC)
**Author:** Forward Deployed Engineer (FDE)
**Target Period:** January 2026 (`yellow_tripdata_2026-01.parquet`)

---

### Business Context & Problem Statement
The NYC TLC manages over 100,000 daily passenger trips. However, source telemetry collected from disparate hardware vendors (VeriFone, CMT, pilot providers) contains substantial anomalies—negative fares, zero-distance rides, unclosed meters, and missing fields. Novice pipelines silently drop up to 30% of data or propagate corrupt metrics into policy decisions.

**FDE Mission:**
1. Map and preserve raw multi-mode data sources (Parquet + REST API Zone Lookup).
2. Profile and explicitly validate trips, quarantining anomalies with audit tags rather than silently dropping.
3. Model the business workflow using a Star Schema relational model.
4. Compute 5 actionable operational KPIs supporting dispatch rebalancing, congestion pricing, and driver earnings.


## 1. Multi-Mode Data Retrieval & Integrity Verification (Class 5)
We retrieve data through two distinct modes:
- **Mode 1 (Parquet File):** Ingest raw trip records, verifying SHA-256 tamper-evident checksums and metadata.
- **Mode 2 (Network HTTP API):** Programmatically download the official TLC Taxi Zone Lookup reference CSV directly from the TLC CloudFront CDN.
- **Mode 3 (Analytical SQL Engine):** Register both datasets in an embedded DuckDB session for zero-copy high-performance transformations.

In [1]:
import sys
from pathlib import Path
import duckdb
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval import DataRetriever
from src.config import RAW_DATA_DIR

retriever = DataRetriever(raw_dir=PROJECT_ROOT / "data" / "raw")
zone_path, zone_meta = retriever.retrieve_taxi_zones_via_api()
trip_path, trip_meta = retriever.retrieve_trips_parquet()
con = retriever.load_into_duckdb(trip_path, zone_path)

print(f"Raw Ingestion Complete!")
print(f"Trips Parquet Rows: {trip_meta['num_rows']:,} | SHA256: {trip_meta['sha256'][:16]}...")
print(f"Taxi Zones Count:   {zone_meta['record_count']} | SHA256: {zone_meta['sha256'][:16]}...")

## 2. Exploratory Data Profiling & Quality Issues (Class 6)
Before applying transformations, we profile the dataset across:
1. **Null Analysis:** Quantifying missing values per column.
2. **Temporal Anomalies:** Identifying negative durations, trips > 24 hours, and dates outside January 2026.
3. **Physical & Speed Anomalies:** Detecting zero distances, trips > 150 miles, and speeds exceeding 85 mph.
4. **Financial Anomalies:** Identifying negative fares, refunds, and zero total charges.

In [2]:
# Profile Null Counts
profile_df = con.execute("""
SELECT 
    COUNT(*) as total_records,
    COUNT(*) - COUNT(passenger_count) as null_passenger_count,
    COUNT(*) - COUNT(RatecodeID) as null_ratecode_id,
    COUNT(*) - COUNT(congestion_surcharge) as null_congestion_surcharge,
    SUM(CASE WHEN tpep_dropoff_datetime < tpep_pickup_datetime THEN 1 ELSE 0 END) as negative_duration,
    SUM(CASE WHEN trip_distance <= 0 THEN 1 ELSE 0 END) as zero_or_neg_distance,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) as negative_fare,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) as negative_total
FROM raw_trips
""").fetchdf()
profile_df.T

### Critical FDE Insight: The 29.2% Missing Passenger / RateCode Anomaly
Notice that `passenger_count`, `RatecodeID`, and `congestion_surcharge` have exactly **1,088,058 missing values** (29.21% of the dataset). Furthermore, all of these correspond to `payment_type = 0` (unspecified/voided dispatch).

> **The FDE Judgement Call:**
> If a data engineer applied a naive `df.dropna()`, they would drop **almost 1.1 million legitimate taxi trips** representing tens of millions of dollars in transit revenue. We retain these records in the curated fact table for operational and revenue metrics, tagging missing categories gracefully as `Unspecified`.

## 3. Data Validation & Anomaly Quarantine Routing (Class 6)
Instead of silently dropping invalid rows, our rule engine:
- Evaluates 10 business validation rules.
- Tags anomalous records with diagnostic failure codes (`OUT_OF_PERIOD`, `NEGATIVE_DURATION`, `DISTANCE_UNDER_MIN`, `SPEED_OVER_85MPH`, etc.).
- Routes clean records to `clean_trips` and quarantined anomalies to `quarantined_trips`.

In [3]:
from src.validation import DataValidator

validator = DataValidator()
con, audit_report = validator.run_validation(con, target_month="2026-01")

print(f"Total Records Evaluated: {audit_report['total_records']:,}")
print(f"Clean Records (Passed):  {audit_report['clean_records']:,} ({audit_report['clean_rate_pct']}%)")
print(f"Quarantined Records:     {audit_report['quarantined_records']:,} ({audit_report['quarantine_rate_pct']}%)")

pd.DataFrame(list(audit_report['violations_by_rule'].items()), columns=['Rule', 'Violation Count'])

## 4. Workflow Modeling & Relational Star Schema (Class 7)
We transform the validated data into a normalized Star Schema:
- **Dimensions:** `dim_zones`, `dim_vendors`, `dim_rate_codes`, `dim_payment_types`
- **Fact Table:** `fct_trips` enriched with trip duration, speed, time windows, and fee flags.

In [4]:
from src.transformation import DataTransformer

transformer = DataTransformer(processed_dir=PROJECT_ROOT / "data" / "processed")
con = transformer.build_dimensional_model(con)

# Preview Enriched Fact Table
sample_fct = con.execute("""
SELECT 
    trip_id, pickup_datetime, time_window, duration_minutes, trip_distance_miles, 
    effective_speed_mph, total_amount, revenue_per_minute, is_airport_trip
FROM fct_trips 
LIMIT 5
""").fetchdf()
sample_fct

## 5. Operational Metrics & Business KPIs (Class 7)
We calculate the 5 core operational KPIs supporting TLC operations and fleet management.

In [5]:
from src.metrics import MetricCalculator

calculator = MetricCalculator(outputs_dir=PROJECT_ROOT / "data" / "outputs")
metrics = calculator.calculate_all_metrics(con, target_month="2026-01")

# Executive KPI Summary
print("=== EXECUTIVE SUMMARY KPIS ===")
metrics['executive_summary_kpis'].T

In [6]:
# Metric 1: Transit Speed & Congestion Penalty
print("=== METRIC 1: SPEED & CONGESTION BY BOROUGH ===")
metrics['metric_1_congestion_speed'].head(10)

In [7]:
# Metric 4: Airport Corridor Performance
print("=== METRIC 4: AIRPORT CORRIDOR PERFORMANCE ===")
metrics['metric_4_airport_corridor']

In [8]:
# Metric 5: Upstream Vendor Data Trust & Hardware Integrity
print("=== METRIC 5: DATA TRUST BY VENDOR ===")
metrics['metric_5_data_trust_vendor']

## 6. Analytical Governance Framework

| Category | Detail |
| :--- | :--- |
| **Known Facts** | 3,724,889 trips processed; 3,463,451 (92.98%) valid; $101.28M total passenger expenditure; 263 official TLC zones. |
| **Unknown Variables** | Driver cruising/idle time between trips (no telemetry); cash tips paid outside the meter; app-hailed vs street-hailed origin. |
| **FDE Assumptions** | 29.2% null passenger counts preserved for revenue/volume metrics; speeds > 85 mph quarantined as sensor errors; trips < 60s treated as meter drop cancellations. |
| **Limitations** | Speeds reflect zone-to-zone straight-line approximations; monthly batch grain rather than real-time event streaming. |

---
**End of Notebook.**